# CELL 1 — LOAD GENOME MANIFEST & BASIC OVERVIEW

In [ ]:

import pandas as pd
import numpy as np
import os

print("=" * 70)
print("LOAD GENOME MANIFEST & BASIC OVERVIEW")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load genome manifest exported from File 01
# ------------------------------------------------------------

FILE_PATH = "/content/genome_manifest.csv"

df_manifest = pd.read_csv(
    FILE_PATH,
    dtype={
        "Genome ID": "string",
        "Genome ID String": "string",
        "Taxon ID": "string"
    }
)

# ------------------------------------------------------------
# 2. Basic dataset information
# ------------------------------------------------------------

print(f"File name            : {os.path.basename(FILE_PATH)}")
print(f"Number of rows       : {df_manifest.shape[0]:,}")
print(f"Number of columns    : {df_manifest.shape[1]:,}")
print(f"Unique Genome IDs    : {df_manifest['Genome ID'].nunique():,}")
print(f"Missing Genome IDs   : {df_manifest['Genome ID'].isna().sum():,}")
print(f"Duplicate Genome IDs : {df_manifest['Genome ID'].duplicated().sum():,}")

# ------------------------------------------------------------
# 3. Column overview
# ------------------------------------------------------------

print("\nColumns:")
for col in df_manifest.columns:
    print(f"- {col}")

# ------------------------------------------------------------
# 4. Preview
# ------------------------------------------------------------

print("\nPreview:")
display(df_manifest.head())

LOAD GENOME MANIFEST & BASIC OVERVIEW
File name            : genome_manifest.csv
Number of rows       : 4,270
Number of columns    : 6
Unique Genome IDs    : 4,270
Missing Genome IDs   : 0
Duplicate Genome IDs : 0

Columns:
- Genome ID
- Genome ID String
- Genome Name
- Taxon ID
- Antibiotic
- Resistant Phenotype

Preview:


,Genome ID,Genome ID String,Genome Name,Taxon ID,Antibiotic,Resistant Phenotype
0,1284787.3,1284787.3,Klebsiella pneumoniae UHKPC40,1284787,meropenem,Resistant
1,1284788.3,1284788.3,Klebsiella pneumoniae UHKPC23,1284788,meropenem,Resistant
2,1284789.4,1284789.4,Klebsiella pneumoniae UHKPC28,1284789,meropenem,Resistant
3,1284790.3,1284790.3,Klebsiella pneumoniae UHKPC05,1284790,meropenem,Resistant
4,1284791.4,1284791.4,Klebsiella pneumoniae UHKPC47,1284791,meropenem,Resistant


# Cell 2 — Genome Manifest Integrity & Target Validation

In [ ]:
# ============================================================
# CELL 2 — GENOME MANIFEST INTEGRITY & TARGET VALIDATION
# ============================================================

print("=" * 70)
print("GENOME MANIFEST INTEGRITY & TARGET VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Kiểm tra các cột bắt buộc
# ------------------------------------------------------------

required_columns = [
    "Genome ID",
    "Genome ID String",
    "Genome Name",
    "Taxon ID",
    "Antibiotic",
    "Resistant Phenotype"
]

missing_required_columns = [
    col for col in required_columns
    if col not in df_manifest.columns
]

print(f"Required columns present     : {len(missing_required_columns) == 0}")

if missing_required_columns:
    print(f"Missing required columns     : {missing_required_columns}")
else:
    print("Missing required columns     : None")


# ------------------------------------------------------------
# 2. Kiểm tra Genome ID
# ------------------------------------------------------------

missing_genome_id = df_manifest["Genome ID"].isna().sum()
duplicate_genome_id = df_manifest["Genome ID"].duplicated().sum()

genome_id_match = (
    df_manifest["Genome ID"]
    .eq(df_manifest["Genome ID String"])
    .sum()
)

valid_genome_id_format = (
    df_manifest["Genome ID String"]
    .str.match(r"^\d+\.\d+$", na=False)
    .sum()
)

print("\n" + "-" * 70)
print("GENOME ID CHECK")
print("-" * 70)

print(f"Missing Genome IDs           : {missing_genome_id:,}")
print(f"Duplicate Genome IDs         : {duplicate_genome_id:,}")
print(
    f"Genome ID == Genome ID String: "
    f"{genome_id_match:,}/{len(df_manifest):,}"
)
print(
    f"Valid Genome ID format       : "
    f"{valid_genome_id_format:,}/{len(df_manifest):,}"
)


# ------------------------------------------------------------
# 3. Kiểm tra target phenotype
# ------------------------------------------------------------

target_counts = (
    df_manifest["Resistant Phenotype"]
    .value_counts(dropna=False)
)

valid_target_labels = {
    "Resistant",
    "Susceptible"
}

observed_target_labels = set(
    df_manifest["Resistant Phenotype"]
    .dropna()
    .unique()
)

unexpected_target_labels = (
    observed_target_labels - valid_target_labels
)

print("\n" + "-" * 70)
print("TARGET CHECK")
print("-" * 70)

print(f"Missing target labels        : "
      f"{df_manifest['Resistant Phenotype'].isna().sum():,}")

print(f"Observed target labels       : "
      f"{sorted(observed_target_labels)}")

print(f"Unexpected target labels     : "
      f"{sorted(unexpected_target_labels) if unexpected_target_labels else 'None'}")

print("\nTarget distribution:")
display(
    target_counts
    .rename_axis("Resistant Phenotype")
    .reset_index(name="Genome Count")
)


# ------------------------------------------------------------
# 4. Kiểm tra antibiotic
# ------------------------------------------------------------

antibiotic_values = (
    df_manifest["Antibiotic"]
    .dropna()
    .str.lower()
    .unique()
)

print("-" * 70)
print("ANTIBIOTIC CHECK")
print("-" * 70)

print(f"Unique antibiotic values     : {antibiotic_values.tolist()}")
print(
    f"All records are meropenem    : "
    f"{df_manifest['Antibiotic'].str.lower().eq('meropenem').all()}"
)


# ------------------------------------------------------------
# 5. Kiểm tra taxonomy / genome name
# ------------------------------------------------------------

kp_name_match = (
    df_manifest["Genome Name"]
    .str.contains(
        "Klebsiella pneumoniae",
        case=False,
        na=False
    )
    .sum()
)

print("\n" + "-" * 70)
print("TAXONOMY CHECK")
print("-" * 70)

print(
    f"Genome names matching "
    f"Klebsiella pneumoniae       : "
    f"{kp_name_match:,}/{len(df_manifest):,}"
)


# ------------------------------------------------------------
# 6. Final manifest validation summary
# ------------------------------------------------------------

manifest_valid = all([
    len(missing_required_columns) == 0,
    missing_genome_id == 0,
    duplicate_genome_id == 0,
    genome_id_match == len(df_manifest),
    valid_genome_id_format == len(df_manifest),
    df_manifest["Resistant Phenotype"].notna().all(),
    len(unexpected_target_labels) == 0,
    df_manifest["Antibiotic"].str.lower().eq("meropenem").all(),
    kp_name_match == len(df_manifest)
])

print("\n" + "=" * 70)
print("FINAL MANIFEST VALIDATION")
print("=" * 70)

print(f"Manifest ready for genome retrieval : {manifest_valid}")

GENOME MANIFEST INTEGRITY & TARGET VALIDATION
Required columns present     : True
Missing required columns     : None

----------------------------------------------------------------------
GENOME ID CHECK
----------------------------------------------------------------------
Missing Genome IDs           : 0
Duplicate Genome IDs         : 0
Genome ID == Genome ID String: 4,270/4,270
Valid Genome ID format       : 4,270/4,270

----------------------------------------------------------------------
TARGET CHECK
----------------------------------------------------------------------
Missing target labels        : 0
Observed target labels       : ['Resistant', 'Susceptible']
Unexpected target labels     : None

Target distribution:


,Resistant Phenotype,Genome Count
0,Susceptible,2661
1,Resistant,1609


----------------------------------------------------------------------
ANTIBIOTIC CHECK
----------------------------------------------------------------------
Unique antibiotic values     : ['meropenem']
All records are meropenem    : True

----------------------------------------------------------------------
TAXONOMY CHECK
----------------------------------------------------------------------
Genome names matching Klebsiella pneumoniae       : 4,270/4,270

FINAL MANIFEST VALIDATION
Manifest ready for genome retrieval : True


# Cell 3 — Pilot BV-BRC Genome Retrieval Check

In [ ]:
# ============================================================
# CELL 3 — PILOT BV-BRC GENOME RETRIEVAL CHECK
# ============================================================

import requests
import time

print("=" * 70)
print("PILOT BV-BRC GENOME RETRIEVAL CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Chọn pilot sample cân bằng theo phenotype
#    5 Resistant + 5 Susceptible
# ------------------------------------------------------------

pilot_resistant = (
    df_manifest[
        df_manifest["Resistant Phenotype"] == "Resistant"
    ]
    .sample(n=5, random_state=42)
)

pilot_susceptible = (
    df_manifest[
        df_manifest["Resistant Phenotype"] == "Susceptible"
    ]
    .sample(n=5, random_state=42)
)

df_pilot = (
    pd.concat(
        [pilot_resistant, pilot_susceptible],
        ignore_index=True
    )
)

print(f"Pilot genomes selected : {len(df_pilot)}")

print("\nPilot phenotype distribution:")
display(
    df_pilot["Resistant Phenotype"]
    .value_counts()
    .rename_axis("Resistant Phenotype")
    .reset_index(name="Genome Count")
)


# ------------------------------------------------------------
# 2. BV-BRC Genome API configuration
# ------------------------------------------------------------

BASE_URL = (
    "https://" +
    "www.bv-brc.org" +
    "/api/genome"
)

HEADERS = {
    "Accept": "application/json",
    "User-Agent": "AMR-Genome-ML-Research/1.0"
}


# ------------------------------------------------------------
# 3. Kiểm tra từng Genome ID
# ------------------------------------------------------------

retrieval_results = []

for _, row in df_pilot.iterrows():

    genome_id = row["Genome ID String"]

    endpoint = f"{BASE_URL}/{genome_id}"

    try:
        response = requests.get(
            endpoint,
            headers=HEADERS,
            timeout=30
        )

        http_status = response.status_code

        if response.ok:

            try:
                record = response.json()

                returned_genome_id = record.get(
                    "genome_id",
                    None
                )

                returned_genome_name = record.get(
                    "genome_name",
                    None
                )

                accessible = True

            except Exception:
                returned_genome_id = None
                returned_genome_name = None
                accessible = False

        else:
            returned_genome_id = None
            returned_genome_name = None
            accessible = False

    except requests.RequestException:
        http_status = None
        returned_genome_id = None
        returned_genome_name = None
        accessible = False

    retrieval_results.append({
        "Genome ID": genome_id,
        "Phenotype": row["Resistant Phenotype"],
        "HTTP Status": http_status,
        "Accessible": accessible,
        "Returned Genome ID": returned_genome_id,
        "Returned Genome Name": returned_genome_name
    })

    # Tránh gửi request quá nhanh
    time.sleep(0.2)


# ------------------------------------------------------------
# 4. Tổng hợp kết quả
# ------------------------------------------------------------

df_pilot_retrieval = pd.DataFrame(
    retrieval_results
)

print("\n" + "-" * 70)
print("PILOT RETRIEVAL RESULTS")
print("-" * 70)

display(df_pilot_retrieval)


# ------------------------------------------------------------
# 5. Retrieval summary
# ------------------------------------------------------------

successful_retrievals = (
    df_pilot_retrieval["Accessible"]
    .sum()
)

failed_retrievals = (
    len(df_pilot_retrieval)
    - successful_retrievals
)

print("\n" + "=" * 70)
print("PILOT RETRIEVAL SUMMARY")
print("=" * 70)

print(
    f"Successfully retrieved : "
    f"{successful_retrievals}/{len(df_pilot_retrieval)}"
)

print(
    f"Failed retrievals      : "
    f"{failed_retrievals}/{len(df_pilot_retrieval)}"
)

print(
    f"Pilot retrieval success: "
    f"{successful_retrievals == len(df_pilot_retrieval)}"
)

PILOT BV-BRC GENOME RETRIEVAL CHECK
Pilot genomes selected : 10

Pilot phenotype distribution:


,Resistant Phenotype,Genome Count
0,Resistant,5
1,Susceptible,5



----------------------------------------------------------------------
PILOT RETRIEVAL RESULTS
----------------------------------------------------------------------


,Genome ID,Phenotype,HTTP Status,Accessible,Returned Genome ID,Returned Genome Name
0,1284819.3,Resistant,200,True,1284819.3,Klebsiella pneumoniae UHKPC45
1,1438749.3,Resistant,200,True,1438749.3,Klebsiella pneumoniae CHS 43
2,573.18090,Resistant,200,True,573.18090,Klebsiella pneumoniae strain AR_0141
3,573.66474,Resistant,200,True,573.66474,Klebsiella pneumoniae 27261_8#98
4,573.13138,Resistant,200,True,573.13138,Klebsiella pneumoniae KPN1137ec
5,573.66631,Susceptible,200,True,573.66631,Klebsiella pneumoniae KP_NORM_BLD_114048
6,573.12989,Susceptible,200,True,573.12989,Klebsiella pneumoniae 46
7,573.14532,Susceptible,200,True,573.14532,Klebsiella pneumoniae KPN968ec
8,573.26303,Susceptible,200,True,573.26303,Klebsiella pneumoniae strain 5012STDY7626423
9,573.24273,Susceptible,200,True,573.24273,Klebsiella pneumoniae strain 275



PILOT RETRIEVAL SUMMARY
Successfully retrieved : 10/10
Failed retrievals      : 0/10
Pilot retrieval success: True


# Cell 4 — Pilot Genome Assembly QC Metadata Audit

In [ ]:
# ============================================================
# CELL 4 — PILOT GENOME ASSEMBLY QC METADATA AUDIT
# ============================================================

print("=" * 70)
print("PILOT GENOME ASSEMBLY QC METADATA AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Các trường metadata dùng để đánh giá assembly / annotation
# ------------------------------------------------------------

qc_fields = [
    "genome_id",
    "genome_name",
    "assembly_accession",
    "assembly_method",
    "genome_length",
    "contigs",
    "contig_n50",
    "contig_l50",
    "gc_content",
    "cds",
    "partial_cds",
    "partial_cds_ratio",
    "fine_consistency"
]


# ------------------------------------------------------------
# 2. Retrieve QC metadata cho 10 pilot genomes
# ------------------------------------------------------------

pilot_qc_records = []

for _, row in df_pilot.iterrows():

    genome_id = row["Genome ID String"]

    endpoint = f"{BASE_URL}/{genome_id}"

    try:
        response = requests.get(
            endpoint,
            headers=HEADERS,
            timeout=30
        )

        if response.ok:

            record = response.json()

            qc_record = {
                field: record.get(field, None)
                for field in qc_fields
            }

            qc_record["Resistant Phenotype"] = (
                row["Resistant Phenotype"]
            )

            qc_record["Retrieval Status"] = "Success"

        else:

            qc_record = {
                field: None
                for field in qc_fields
            }

            qc_record["genome_id"] = genome_id
            qc_record["Resistant Phenotype"] = (
                row["Resistant Phenotype"]
            )

            qc_record["Retrieval Status"] = (
                f"HTTP {response.status_code}"
            )

    except requests.RequestException as e:

        qc_record = {
            field: None
            for field in qc_fields
        }

        qc_record["genome_id"] = genome_id
        qc_record["Resistant Phenotype"] = (
            row["Resistant Phenotype"]
        )

        qc_record["Retrieval Status"] = "Request Error"

    pilot_qc_records.append(qc_record)

    time.sleep(0.2)


# ------------------------------------------------------------
# 3. Tạo bảng QC metadata
# ------------------------------------------------------------

df_pilot_qc = pd.DataFrame(
    pilot_qc_records
)

print("\nPilot QC metadata:")
display(df_pilot_qc)


# ------------------------------------------------------------
# 4. Kiểm tra tỷ lệ dữ liệu có sẵn cho từng QC field
# ------------------------------------------------------------

qc_availability = pd.DataFrame({
    "Non-null Count": df_pilot_qc[qc_fields].notna().sum(),
    "Missing Count": df_pilot_qc[qc_fields].isna().sum(),
    "Missing (%)": (
        df_pilot_qc[qc_fields]
        .isna()
        .mean()
        * 100
    ).round(2)
})

qc_availability = (
    qc_availability
    .sort_values(
        "Missing (%)",
        ascending=False
    )
)

print("\n" + "-" * 70)
print("QC METADATA AVAILABILITY")
print("-" * 70)

display(qc_availability)


# ------------------------------------------------------------
# 5. Basic numeric summary
# ------------------------------------------------------------

numeric_qc_fields = [
    "genome_length",
    "contigs",
    "contig_n50",
    "contig_l50",
    "gc_content",
    "cds",
    "partial_cds",
    "partial_cds_ratio",
    "fine_consistency"
]

print("\n" + "-" * 70)
print("PILOT NUMERIC QC SUMMARY")
print("-" * 70)

display(
    df_pilot_qc[numeric_qc_fields]
    .describe()
    .T
    .round(2)
)

PILOT GENOME ASSEMBLY QC METADATA AUDIT

Pilot QC metadata:


,genome_id,genome_name,assembly_accession,assembly_method,genome_length,contigs,contig_n50,contig_l50,gc_content,cds,partial_cds,partial_cds_ratio,fine_consistency,Resistant Phenotype,Retrieval Status
0,1284819.3,Klebsiella pneumoniae UHKPC45,GCA_000409005.1,Newbler v. 2.3,5620790,210,95425,20,57.080000,5503,0,0,97.90,Resistant,Success
1,1438749.3,Klebsiella pneumoniae CHS 43,GCA_000693815.1,allpaths v. R48559,5684653,40,344765,6,57.100000,5441,0,0,97.30,Resistant,Success
2,573.18090,Klebsiella pneumoniae strain AR_0141,GCA_003071305.1,HGAP v. 2.3,5574787,3,5225737,1,57.267338,5741,0,0,98.10,Resistant,Success
3,573.66474,Klebsiella pneumoniae 27261_8#98,None,None,5658955,137,260020,8,57.051520,5588,0,0,98.00,Resistant,Success
4,573.13138,Klebsiella pneumoniae KPN1137ec,None,None,5437697,123,145826,13,57.220000,5491,0,0,98.00,Resistant,Success
5,573.66631,Klebsiella pneumoniae KP_NORM_BLD_114048,None,None,5440410,88,197655,10,57.397823,5331,0,0,97.90,Susceptible,Success
6,573.12989,Klebsiella pneumoniae 46,None,None,5723650,538,186038,9,57.140000,5822,0,0,98.00,Susceptible,Success
7,573.14532,Klebsiella pneumoniae KPN968ec,None,None,5374324,76,165345,11,57.390000,5281,0,0,98.20,Susceptible,Success
8,573.26303,Klebsiella pneumoniae strain 5012STDY7626423,GCA_900776355.1,None,5502314,48,528132,4,57.157368,5373,0,0,93.61,Susceptible,Success
9,573.24273,Klebsiella pneumoniae strain 275,None,None,5273288,92,309587,5,57.478560,5150,0,0,93.97,Susceptible,Success



----------------------------------------------------------------------
QC METADATA AVAILABILITY
----------------------------------------------------------------------


,Non-null Count,Missing Count,Missing (%)
assembly_method,3,7,70.0
assembly_accession,4,6,60.0
genome_id,10,0,0.0
genome_name,10,0,0.0
genome_length,10,0,0.0
contigs,10,0,0.0
contig_n50,10,0,0.0
contig_l50,10,0,0.0
gc_content,10,0,0.0
cds,10,0,0.0



----------------------------------------------------------------------
PILOT NUMERIC QC SUMMARY
----------------------------------------------------------------------


,count,mean,std,min,25%,50%,75%,max
genome_length,10.0,5529086.80,147473.70,5273288.00,5438375.25,5538550.50,5649413.75,5723650.00
contigs,10.0,135.50,152.66,3.00,55.00,90.00,133.50,538.00
contig_n50,10.0,745853.00,1578958.48,95425.00,170518.25,228837.50,335970.50,5225737.00
contig_l50,10.0,8.70,5.33,1.00,5.25,8.50,10.75,20.00
gc_content,10.0,57.23,0.15,57.05,57.11,57.19,57.36,57.48
cds,10.0,5472.10,205.60,5150.00,5341.50,5466.00,5566.75,5822.00
partial_cds,10.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
partial_cds_ratio,10.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
fine_consistency,10.0,97.10,1.76,93.61,97.45,97.95,98.00,98.20


# Cell 5 — Full Genome QC Metadata Retrieval

In [ ]:
# ============================================================
# CELL 5 — FULL GENOME QC METADATA RETRIEVAL
# ============================================================

import requests
import time
from tqdm.auto import tqdm

print("=" * 70)
print("FULL GENOME QC METADATA RETRIEVAL")
print("=" * 70)

# ------------------------------------------------------------
# 1. BV-BRC fields cần lấy
# ------------------------------------------------------------

full_qc_fields = [
    "genome_id",
    "genome_name",
    "assembly_accession",
    "assembly_method",
    "genome_length",
    "contigs",
    "contig_n50",
    "contig_l50",
    "gc_content",
    "cds",
    "partial_cds",
    "partial_cds_ratio",
    "fine_consistency"
]

# ------------------------------------------------------------
# 2. Chuẩn bị Genome IDs
# ------------------------------------------------------------

genome_ids = (
    df_manifest["Genome ID String"]
    .astype(str)
    .tolist()
)

print(f"Genome IDs to retrieve : {len(genome_ids):,}")

# Batch nhỏ để URL không quá dài
BATCH_SIZE = 100

genome_id_batches = [
    genome_ids[i:i + BATCH_SIZE]
    for i in range(0, len(genome_ids), BATCH_SIZE)
]

print(f"Number of batches      : {len(genome_id_batches):,}")
print(f"Batch size             : {BATCH_SIZE}")


# ------------------------------------------------------------
# 3. Hàm retrieve một batch từ BV-BRC
# ------------------------------------------------------------

def retrieve_genome_qc_batch(
    batch_ids,
    max_retries=3
):

    id_string = ",".join(batch_ids)

    field_string = ",".join(full_qc_fields)

    query = (
        f"in(genome_id,({id_string}))"
        f"&select({field_string})"
        f"&limit({len(batch_ids)})"
    )

    url = (
        "https://www.bv-brc.org/api/genome/"
        f"?{query}"
    )

    headers = {
        "Accept": "application/json",
        "User-Agent": "AMR-Genome-ML-Research/1.0"
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=60
            )

            if response.status_code == 200:
                return response.json(), None

            # Retry khi server tạm thời quá tải
            if response.status_code in [
                429,
                500,
                502,
                503,
                504
            ]:
                time.sleep(attempt * 2)
                continue

            return None, (
                f"HTTP {response.status_code}"
            )

        except requests.RequestException as e:

            if attempt < max_retries:
                time.sleep(attempt * 2)
                continue

            return None, str(e)

    return None, "Unknown retrieval error"


# ------------------------------------------------------------
# 4. Retrieve toàn bộ QC metadata
# ------------------------------------------------------------

all_qc_records = []
failed_batches = []

for batch_index, batch_ids in enumerate(
    tqdm(
        genome_id_batches,
        desc="Retrieving genome QC metadata"
    ),
    start=1
):

    records, error = retrieve_genome_qc_batch(
        batch_ids
    )

    if records is not None:
        all_qc_records.extend(records)

    else:
        failed_batches.append({
            "Batch": batch_index,
            "Genome Count": len(batch_ids),
            "Error": error
        })

    # Tránh gửi request quá nhanh
    time.sleep(0.2)


# ------------------------------------------------------------
# 5. Tạo DataFrame
# ------------------------------------------------------------

df_genome_qc = pd.DataFrame(
    all_qc_records
)

print("\n" + "-" * 70)
print("RETRIEVAL SUMMARY")
print("-" * 70)

print(
    f"Requested genomes       : "
    f"{len(genome_ids):,}"
)

print(
    f"Returned records        : "
    f"{len(df_genome_qc):,}"
)

print(
    f"Unique returned genomes : "
    f"{df_genome_qc['genome_id'].nunique():,}"
    if "genome_id" in df_genome_qc.columns
    else "Unique returned genomes : 0"
)

print(
    f"Failed batches          : "
    f"{len(failed_batches):,}"
)


# ------------------------------------------------------------
# 6. Kiểm tra Genome IDs bị thiếu sau retrieval
# ------------------------------------------------------------

returned_genome_ids = set(
    df_genome_qc["genome_id"]
    .astype(str)
) if "genome_id" in df_genome_qc.columns else set()

requested_genome_ids = set(
    genome_ids
)

missing_after_retrieval = (
    requested_genome_ids
    - returned_genome_ids
)

unexpected_genome_ids = (
    returned_genome_ids
    - requested_genome_ids
)

print(
    f"Missing after retrieval : "
    f"{len(missing_after_retrieval):,}"
)

print(
    f"Unexpected Genome IDs   : "
    f"{len(unexpected_genome_ids):,}"
)


# ------------------------------------------------------------
# 7. Hiển thị failed batches nếu có
# ------------------------------------------------------------

if failed_batches:

    print("\nFailed batches:")

    display(
        pd.DataFrame(failed_batches)
    )


# ------------------------------------------------------------
# 8. Metadata availability toàn cohort
# ------------------------------------------------------------

if len(df_genome_qc) > 0:

    qc_availability_full = pd.DataFrame({
        "Non-null Count":
            df_genome_qc[full_qc_fields]
            .notna()
            .sum(),

        "Missing Count":
            df_genome_qc[full_qc_fields]
            .isna()
            .sum(),

        "Missing (%)":
            (
                df_genome_qc[full_qc_fields]
                .isna()
                .mean()
                * 100
            ).round(2)
    })

    qc_availability_full = (
        qc_availability_full
        .sort_values(
            "Missing (%)",
            ascending=False
        )
    )

    print("\n" + "-" * 70)
    print("FULL QC METADATA AVAILABILITY")
    print("-" * 70)

    display(qc_availability_full)


# ------------------------------------------------------------
# 9. Numeric QC overview
# ------------------------------------------------------------

numeric_qc_fields = [
    "genome_length",
    "contigs",
    "contig_n50",
    "contig_l50",
    "gc_content",
    "cds",
    "partial_cds",
    "partial_cds_ratio",
    "fine_consistency"
]

available_numeric_fields = [
    col for col in numeric_qc_fields
    if col in df_genome_qc.columns
]

print("\n" + "-" * 70)
print("FULL NUMERIC QC SUMMARY")
print("-" * 70)

display(
    df_genome_qc[
        available_numeric_fields
    ]
    .describe()
    .T
    .round(2)
)

FULL GENOME QC METADATA RETRIEVAL
Genome IDs to retrieve : 4,270
Number of batches      : 43
Batch size             : 100


Retrieving genome QC metadata:   0%|          | 0/43 [00:00<?, ?it/s]


----------------------------------------------------------------------
RETRIEVAL SUMMARY
----------------------------------------------------------------------
Requested genomes       : 4,270
Returned records        : 4,270
Unique returned genomes : 4,270
Failed batches          : 0
Missing after retrieval : 0
Unexpected Genome IDs   : 0

----------------------------------------------------------------------
FULL QC METADATA AVAILABILITY
----------------------------------------------------------------------


,Non-null Count,Missing Count,Missing (%)
assembly_method,310,3960,92.74
assembly_accession,955,3315,77.63
genome_id,4270,0,0.00
genome_name,4270,0,0.00
genome_length,4270,0,0.00
contigs,4270,0,0.00
contig_n50,4270,0,0.00
contig_l50,4270,0,0.00
gc_content,4270,0,0.00
cds,4270,0,0.00



----------------------------------------------------------------------
FULL NUMERIC QC SUMMARY
----------------------------------------------------------------------


,count,mean,std,min,25%,50%,75%,max
genome_length,4270.0,5552647.25,311667.81,17021.00,5428599.25,5522748.50,5638805.00,11147041.00
contigs,4270.0,119.49,159.77,1.00,70.00,93.00,128.00,5155.00
contig_n50,4270.0,362718.16,766740.06,352.00,146314.00,209571.00,309940.00,5633878.00
contig_l50,4270.0,10.71,14.38,0.00,6.00,9.00,13.00,490.00
gc_content,4270.0,57.17,0.41,46.35,57.08,57.21,57.33,60.48
cds,4270.0,5525.35,385.55,46.00,5356.00,5492.00,5645.75,11674.00
partial_cds,4270.0,0.81,7.65,0.00,0.00,0.00,0.00,92.00
partial_cds_ratio,4270.0,0.00,0.00,0.00,0.00,0.00,0.00,0.02
fine_consistency,4270.0,96.86,3.25,39.90,97.30,97.80,98.10,99.00


# Cell 6 — Genome QC Distribution & Extreme Value Audit

In [ ]:
# ============================================================
# CELL 6 — GENOME QC DISTRIBUTION & EXTREME VALUE AUDIT
# ============================================================

print("=" * 70)
print("GENOME QC DISTRIBUTION & EXTREME VALUE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Gắn phenotype vào bảng QC
# ------------------------------------------------------------

df_qc_audit = (
    df_genome_qc
    .merge(
        df_manifest[
            [
                "Genome ID String",
                "Resistant Phenotype"
            ]
        ],
        left_on="genome_id",
        right_on="Genome ID String",
        how="left",
        validate="one_to_one"
    )
    .drop(columns=["Genome ID String"])
)

print(f"QC records             : {len(df_qc_audit):,}")
print(
    f"Missing phenotype      : "
    f"{df_qc_audit['Resistant Phenotype'].isna().sum():,}"
)


# ------------------------------------------------------------
# 2. Các QC metrics chính
# ------------------------------------------------------------

core_qc_fields = [
    "genome_length",
    "contigs",
    "contig_n50",
    "contig_l50",
    "gc_content",
    "cds",
    "fine_consistency"
]

# Đảm bảo các trường được xử lý dạng numeric
for col in core_qc_fields:
    df_qc_audit[col] = pd.to_numeric(
        df_qc_audit[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# 3. Quantile distribution
# ------------------------------------------------------------

quantiles = [
    0.000,
    0.001,
    0.005,
    0.010,
    0.050,
    0.250,
    0.500,
    0.750,
    0.950,
    0.990,
    0.995,
    0.999,
    1.000
]

qc_quantiles = (
    df_qc_audit[core_qc_fields]
    .quantile(quantiles)
    .T
)

qc_quantiles.columns = [
    "Min",
    "0.1%",
    "0.5%",
    "1%",
    "5%",
    "25%",
    "50%",
    "75%",
    "95%",
    "99%",
    "99.5%",
    "99.9%",
    "Max"
]

print("\n" + "-" * 70)
print("QC METRIC QUANTILES")
print("-" * 70)

display(
    qc_quantiles.round(2)
)


# ------------------------------------------------------------
# 4. QC summary theo phenotype
# ------------------------------------------------------------

phenotype_qc_summary = (
    df_qc_audit
    .groupby("Resistant Phenotype")[core_qc_fields]
    .agg(["median", "mean", "min", "max"])
    .round(2)
)

print("\n" + "-" * 70)
print("QC SUMMARY BY RESISTANT PHENOTYPE")
print("-" * 70)

display(
    phenotype_qc_summary
)


# ------------------------------------------------------------
# 5. Hàm hiển thị các genome cực trị
# ------------------------------------------------------------

def show_extremes(
    dataframe,
    column,
    n=10,
    ascending=True
):

    columns_to_show = [
        "genome_id",
        "genome_name",
        "Resistant Phenotype",
        "genome_length",
        "contigs",
        "contig_n50",
        "contig_l50",
        "gc_content",
        "cds",
        "fine_consistency"
    ]

    return (
        dataframe
        .sort_values(
            column,
            ascending=ascending
        )
        [columns_to_show]
        .head(n)
    )


# ------------------------------------------------------------
# 6. Genome length extremes
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOWEST GENOME LENGTH")
print("-" * 70)

display(
    show_extremes(
        df_qc_audit,
        "genome_length",
        n=10,
        ascending=True
    )
)

print("\nHIGHEST GENOME LENGTH")

display(
    show_extremes(
        df_qc_audit,
        "genome_length",
        n=10,
        ascending=False
    )
)


# ------------------------------------------------------------
# 7. Contig count extremes
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("HIGHEST CONTIG COUNTS")
print("-" * 70)

display(
    show_extremes(
        df_qc_audit,
        "contigs",
        n=10,
        ascending=False
    )
)


# ------------------------------------------------------------
# 8. N50 extremes
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOWEST CONTIG N50")
print("-" * 70)

display(
    show_extremes(
        df_qc_audit,
        "contig_n50",
        n=10,
        ascending=True
    )
)


# ------------------------------------------------------------
# 9. GC content extremes
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOWEST GC CONTENT")
print("-" * 70)

display(
    show_extremes(
        df_qc_audit,
        "gc_content",
        n=10,
        ascending=True
    )
)

print("\nHIGHEST GC CONTENT")

display(
    show_extremes(
        df_qc_audit,
        "gc_content",
        n=10,
        ascending=False
    )
)


# ------------------------------------------------------------
# 10. Fine consistency extremes
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOWEST FINE CONSISTENCY")
print("-" * 70)

display(
    show_extremes(
        df_qc_audit,
        "fine_consistency",
        n=10,
        ascending=True
    )
)


# ------------------------------------------------------------
# 11. CDS extremes
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOWEST CDS COUNT")
print("-" * 70)

display(
    show_extremes(
        df_qc_audit,
        "cds",
        n=10,
        ascending=True
    )
)

print("\nHIGHEST CDS COUNT")

display(
    show_extremes(
        df_qc_audit,
        "cds",
        n=10,
        ascending=False
    )
)

GENOME QC DISTRIBUTION & EXTREME VALUE AUDIT
QC records             : 4,270
Missing phenotype      : 0

----------------------------------------------------------------------
QC METRIC QUANTILES
----------------------------------------------------------------------


,Min,0.1%,0.5%,1%,5%,25%,50%,75%,95%,99%,99.5%,99.9%,Max
genome_length,17021.00,4794866.56,5079504.25,5129848.32,5284138.55,5428599.25,5522748.50,5638805.00,5857029.05,6062417.11,6620306.61,9312520.43,11147041.00
contigs,1.00,3.00,4.00,5.00,33.00,70.00,93.00,128.00,270.65,593.00,1090.82,1737.81,5155.00
contig_n50,352.00,7010.22,30732.93,42784.71,93355.15,146314.00,209571.00,309940.00,722806.30,5338758.90,5382973.15,5513343.58,5633878.00
contig_l50,0.00,1.00,1.00,1.00,3.00,6.00,9.00,13.00,20.00,37.31,56.00,196.87,490.00
gc_content,46.35,51.39,55.81,56.41,56.66,57.08,57.21,57.33,57.50,57.74,57.79,58.24,60.48
cds,46.00,4727.07,4919.07,4981.14,5169.00,5356.00,5492.00,5645.75,5901.00,6236.58,6940.13,10599.31,11674.00
fine_consistency,39.90,52.41,75.74,91.58,93.43,97.30,97.80,98.10,98.50,98.70,98.70,98.70,99.00



----------------------------------------------------------------------
QC SUMMARY BY RESISTANT PHENOTYPE
----------------------------------------------------------------------


genome_length                                contigs  \
                           median        mean      min       max  median   
Resistant Phenotype                                                        
Resistant               5596956.0  5627348.76  5240758  11147041   107.0   
Susceptible             5478436.0  5507478.24    17021  11112536    85.0   

                                      contig_n50             ... gc_content  \
                       mean min   max     median       mean  ...        min   
Resistant Phenotype                                          ...              
Resistant            118.26   2  1744   223902.0  425839.38  ...      46.35   
Susceptible          120.24   1  5155   203129.0  324551.29  ...      49.39   

                               cds                       fine_consistency  \
                       max  median     mean   min    max           median   
Resistant Phenotype                                                         
Resistant            60.48  5600.0  5622.92  5085  11674             97.8   
Susceptible          59.67  5429.0  5466.36    46  11229             97.8   

                                        
                      mean   min   max  
Resistant Phenotype                     
Resistant            96.79  39.9  98.7  
Susceptible          96.91  51.4  99.0  

[2 rows x 28 columns]


----------------------------------------------------------------------
LOWEST GENOME LENGTH
----------------------------------------------------------------------


...

,genome_id,genome_name,Resistant Phenotype,genome_length,contigs,contig_n50,contig_l50,gc_content,cds,fine_consistency
2704,573.31080,Klebsiella pneumoniae strain WU21,Resistant,11147041,1715,14403,220,56.560265,11674,39.90
1806,573.15450,Klebsiella pneumoniae strain 10315_6#32,Susceptible,10834068,593,46275,73,53.600000,11113,51.40
274,573.12987,Klebsiella pneumoniae 3386,Susceptible,8354191,3848,100473,18,51.810000,10217,51.70
2561,573.29887,Klebsiella pneumoniae strain Kpngiani7132577,Resistant,8827887,118,256976,11,51.250305,8932,52.26
1083,573.13835,Klebsiella pneumoniae KPN2005ec,Susceptible,8750302,2412,5163,424,57.550000,10212,52.30
1919,573.15570,Klebsiella pneumoniae strain KSB1_4J,Susceptible,9866087,1611,34682,36,54.280000,11229,52.70
613,573.13345,Klebsiella pneumoniae KPN139ec,Susceptible,8862200,1209,31009,64,50.760000,9139,54.50
620,573.13353,Klebsiella pneumoniae KPN140ec,Susceptible,8883036,1128,31829,52,50.700000,9106,54.70
244,573.12957,Klebsiella pneumoniae 3252,Susceptible,7355114,5155,99606,22,55.740000,10740,55.10
2792,573.46106,Klebsiella pneumoniae KSB1_8H,Susceptible,11112536,141,162488,17,54.855984,10954,55.39



----------------------------------------------------------------------
LOWEST CDS COUNT
----------------------------------------------------------------------


,genome_id,genome_name,Resistant Phenotype,genome_length,contigs,contig_n50,contig_l50,gc_content,cds,fine_consistency
2772,573.46086,Klebsiella pneumoniae KSB1_3A,Susceptible,17021,47,352,21,59.673344,46,94.09
2683,573.31059,Klebsiella pneumoniae strain PH10,Susceptible,1989615,163,19881,32,56.964344,2012,78.06
985,573.13732,Klebsiella pneumoniae KPN1869ec,Susceptible,2750007,33,115313,9,57.650000,2632,80.80
2797,573.46111,Klebsiella pneumoniae KSB2_10A,Susceptible,4766596,32,771668,3,55.418648,4587,93.79
3523,573.66705,Klebsiella pneumoniae KP_NORM_BLD_116236,Susceptible,4908248,51,198427,10,57.673470,4719,97.50
3453,573.66634,Klebsiella pneumoniae KP_NORM_BLD_114294,Susceptible,4936999,39,365486,6,57.842910,4749,98.00
2708,573.31084,Klebsiella pneumoniae strain WU9,Susceptible,4978837,65,173098,10,57.981407,4761,94.27
3761,573.66950,Klebsiella pneumoniae KP_NORM_BLD_2260,Susceptible,4949809,120,86237,18,57.778290,4804,98.30
3506,573.66688,Klebsiella pneumoniae KP_NORM_BLD_115984,Susceptible,4982854,33,343689,6,57.717445,4808,97.30
3495,573.66677,Klebsiella pneumoniae KP_NORM_BLD_115414,Susceptible,5055541,40,371837,6,57.749880,4863,98.70



HIGHEST CDS COUNT


,genome_id,genome_name,Resistant Phenotype,genome_length,contigs,contig_n50,contig_l50,gc_content,cds,fine_consistency
2704,573.31080,Klebsiella pneumoniae strain WU21,Resistant,11147041,1715,14403,220,56.560265,11674,39.90
1919,573.15570,Klebsiella pneumoniae strain KSB1_4J,Susceptible,9866087,1611,34682,36,54.280000,11229,52.70
1806,573.15450,Klebsiella pneumoniae strain 10315_6#32,Susceptible,10834068,593,46275,73,53.600000,11113,51.40
2792,573.46106,Klebsiella pneumoniae KSB1_8H,Susceptible,11112536,141,162488,17,54.855984,10954,55.39
244,573.12957,Klebsiella pneumoniae 3252,Susceptible,7355114,5155,99606,22,55.740000,10740,55.10
274,573.12987,Klebsiella pneumoniae 3386,Susceptible,8354191,3848,100473,18,51.810000,10217,51.70
1083,573.13835,Klebsiella pneumoniae KPN2005ec,Susceptible,8750302,2412,5163,424,57.550000,10212,52.30
1070,573.13822,Klebsiella pneumoniae KPN1989ec,Susceptible,8614068,1747,15828,97,54.740000,10163,66.30
2040,573.2024,Klebsiella pneumoniae strain 143500523,Resistant,8610769,1526,66335,27,60.480000,9927,69.50
1844,573.15492,Klebsiella pneumoniae strain 10315_6#85,Susceptible,9470566,276,82386,31,49.390000,9601,61.50


# Cell 7 — Extended BV-BRC Genome Quality Metadata Audit

In [ ]:
# ============================================================
# CELL 7 — EXTENDED BV-BRC GENOME QUALITY METADATA AUDIT
# ============================================================

print("=" * 70)
print("EXTENDED BV-BRC GENOME QUALITY METADATA AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Các QC fields bổ sung từ BV-BRC
# ------------------------------------------------------------

extended_qc_fields = [
    "genome_id",
    "genome_quality",
    "genome_quality_flags",
    "checkm_completeness",
    "checkm_contamination",
    "coarse_consistency"
]

print("Additional QC fields:")
for field in extended_qc_fields:
    print(f"- {field}")


# ------------------------------------------------------------
# 2. Hàm retrieve một batch
# ------------------------------------------------------------

def retrieve_extended_qc_batch(
    batch_ids,
    max_retries=3
):

    id_string = ",".join(batch_ids)
    field_string = ",".join(extended_qc_fields)

    query = (
        f"in(genome_id,({id_string}))"
        f"&select({field_string})"
        f"&limit({len(batch_ids)})"
    )

    url = (
        "https://www.bv-brc.org/api/genome/"
        f"?{query}"
    )

    headers = {
        "Accept": "application/json",
        "User-Agent": "AMR-Genome-ML-Research/1.0"
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=60
            )

            if response.status_code == 200:
                return response.json(), None

            if response.status_code in [
                429,
                500,
                502,
                503,
                504
            ]:
                time.sleep(attempt * 2)
                continue

            return None, (
                f"HTTP {response.status_code}"
            )

        except requests.RequestException as e:

            if attempt < max_retries:
                time.sleep(attempt * 2)
                continue

            return None, str(e)

    return None, "Unknown retrieval error"


# ------------------------------------------------------------
# 3. Retrieve extended QC metadata cho toàn bộ cohort
# ------------------------------------------------------------

extended_qc_records = []
failed_extended_batches = []

for batch_index, batch_ids in enumerate(
    tqdm(
        genome_id_batches,
        desc="Retrieving extended QC metadata"
    ),
    start=1
):

    records, error = retrieve_extended_qc_batch(
        batch_ids
    )

    if records is not None:
        extended_qc_records.extend(records)

    else:
        failed_extended_batches.append({
            "Batch": batch_index,
            "Genome Count": len(batch_ids),
            "Error": error
        })

    time.sleep(0.2)


# ------------------------------------------------------------
# 4. Tạo DataFrame
# ------------------------------------------------------------

df_extended_qc = pd.DataFrame(
    extended_qc_records
)

print("\n" + "-" * 70)
print("EXTENDED QC RETRIEVAL SUMMARY")
print("-" * 70)

print(
    f"Requested genomes       : "
    f"{len(genome_ids):,}"
)

print(
    f"Returned records        : "
    f"{len(df_extended_qc):,}"
)

print(
    f"Unique returned genomes : "
    f"{df_extended_qc['genome_id'].nunique():,}"
    if "genome_id" in df_extended_qc.columns
    else "Unique returned genomes : 0"
)

print(
    f"Failed batches          : "
    f"{len(failed_extended_batches):,}"
)


# ------------------------------------------------------------
# 5. Kiểm tra ID coverage
# ------------------------------------------------------------

returned_extended_ids = set(
    df_extended_qc["genome_id"].astype(str)
)

missing_extended_ids = (
    set(genome_ids)
    - returned_extended_ids
)

print(
    f"Missing after retrieval : "
    f"{len(missing_extended_ids):,}"
)


# ------------------------------------------------------------
# 6. Metadata availability
# ------------------------------------------------------------

extended_availability = pd.DataFrame({
    "Non-null Count":
        df_extended_qc[extended_qc_fields]
        .notna()
        .sum(),

    "Missing Count":
        df_extended_qc[extended_qc_fields]
        .isna()
        .sum(),

    "Missing (%)":
        (
            df_extended_qc[extended_qc_fields]
            .isna()
            .mean()
            * 100
        ).round(2)
})

extended_availability = (
    extended_availability
    .sort_values(
        "Missing (%)",
        ascending=False
    )
)

print("\n" + "-" * 70)
print("EXTENDED QC METADATA AVAILABILITY")
print("-" * 70)

display(extended_availability)


# ------------------------------------------------------------
# 7. Genome quality label distribution
# ------------------------------------------------------------

if "genome_quality" in df_extended_qc.columns:

    print("\n" + "-" * 70)
    print("BV-BRC GENOME QUALITY DISTRIBUTION")
    print("-" * 70)

    display(
        df_extended_qc["genome_quality"]
        .fillna("Missing")
        .value_counts(dropna=False)
        .rename_axis("Genome Quality")
        .reset_index(name="Genome Count")
    )


# ------------------------------------------------------------
# 8. Genome quality flags distribution
# ------------------------------------------------------------

if "genome_quality_flags" in df_extended_qc.columns:

    print("\n" + "-" * 70)
    print("BV-BRC GENOME QUALITY FLAGS")
    print("-" * 70)

    quality_flags = (
        df_extended_qc[
            "genome_quality_flags"
        ]
        .apply(
            lambda x:
                "None"
                if x is None
                else str(x)
        )
        .value_counts()
        .head(20)
        .rename_axis("Genome Quality Flags")
        .reset_index(name="Genome Count")
    )

    display(quality_flags)


# ------------------------------------------------------------
# 9. Numeric extended QC summary
# ------------------------------------------------------------

extended_numeric_fields = [
    "checkm_completeness",
    "checkm_contamination",
    "coarse_consistency"
]

for col in extended_numeric_fields:

    if col in df_extended_qc.columns:

        df_extended_qc[col] = pd.to_numeric(
            df_extended_qc[col],
            errors="coerce"
        )


available_extended_numeric = [
    col for col in extended_numeric_fields
    if col in df_extended_qc.columns
]

print("\n" + "-" * 70)
print("EXTENDED NUMERIC QC SUMMARY")
print("-" * 70)

display(
    df_extended_qc[
        available_extended_numeric
    ]
    .describe(
        percentiles=[
            0.001,
            0.005,
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
    .T
    .round(2)
)


# ------------------------------------------------------------
# 10. Merge vào bảng QC audit hiện tại
# ------------------------------------------------------------

df_qc_audit = (
    df_qc_audit
    .merge(
        df_extended_qc,
        on="genome_id",
        how="left",
        validate="one_to_one"
    )
)

print("\n" + "=" * 70)
print("EXTENDED QC MERGE SUMMARY")
print("=" * 70)

print(
    f"Final QC audit records : "
    f"{len(df_qc_audit):,}"
)

print(
    f"Unique Genome IDs      : "
    f"{df_qc_audit['genome_id'].nunique():,}"
)

EXTENDED BV-BRC GENOME QUALITY METADATA AUDIT
Additional QC fields:
- genome_id
- genome_quality
- genome_quality_flags
- checkm_completeness
- checkm_contamination
- coarse_consistency


Retrieving extended QC metadata:   0%|          | 0/43 [00:00<?, ?it/s]


----------------------------------------------------------------------
EXTENDED QC RETRIEVAL SUMMARY
----------------------------------------------------------------------
Requested genomes       : 4,270
Returned records        : 4,270
Unique returned genomes : 4,270
Failed batches          : 0
Missing after retrieval : 0

----------------------------------------------------------------------
EXTENDED QC METADATA AVAILABILITY
----------------------------------------------------------------------


,Non-null Count,Missing Count,Missing (%)
genome_quality_flags,37,4233,99.13
genome_id,4270,0,0.00
genome_quality,4270,0,0.00
checkm_completeness,4270,0,0.00
checkm_contamination,4270,0,0.00
coarse_consistency,4270,0,0.00



----------------------------------------------------------------------
BV-BRC GENOME QUALITY DISTRIBUTION
----------------------------------------------------------------------


...


----------------------------------------------------------------------
BV-BRC GENOME QUALITY FLAGS
----------------------------------------------------------------------


,Genome Quality Flags,Genome Count
0,nan,4233
1,['Too many contigs'],21
2,['High CheckM contamination score'],11
3,"['Too many contigs', 'Low contig N50']",3
4,['Low CheckM completeness score'],1
5,"['Low contig N50', 'Genome too short', 'Abnorm...",1



----------------------------------------------------------------------
EXTENDED NUMERIC QC SUMMARY
----------------------------------------------------------------------


,count,mean,std,min,0.1%,0.5%,1%,5%,50%,95%,99%,99.5%,99.9%,max
checkm_completeness,4270.0,99.80,2.34,0.00,60.39,96.81,98.00,99.70,100.0,100.0,100.00,100.00,100.00,100.0
checkm_contamination,4270.0,0.38,2.82,0.00,0.00,0.00,0.00,0.00,0.0,0.8,4.32,16.86,47.72,51.3
coarse_consistency,4270.0,98.62,1.80,76.73,83.43,94.47,94.94,95.24,99.4,99.6,99.70,99.70,99.80,99.9



EXTENDED QC MERGE SUMMARY
Final QC audit records : 4,270
Unique Genome IDs      : 4,270


# Cell 8 — BV-BRC Genome Quality Filter Impact Audit

In [ ]:
# ============================================================
# CELL 8 — BV-BRC GENOME QUALITY FILTER IMPACT AUDIT
# ============================================================

print("=" * 70)
print("BV-BRC GENOME QUALITY FILTER IMPACT AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Chuẩn hóa genome quality label
# ------------------------------------------------------------

df_qc_audit["genome_quality"] = (
    df_qc_audit["genome_quality"]
    .astype("string")
    .str.strip()
)

quality_counts = (
    df_qc_audit["genome_quality"]
    .value_counts(dropna=False)
)

print("Genome quality distribution:")

display(
    quality_counts
    .rename_axis("Genome Quality")
    .reset_index(name="Genome Count")
)


# ------------------------------------------------------------
# 2. Tạo candidate QC status
#    Chưa loại genome khỏi dataset
# ------------------------------------------------------------

df_qc_audit["QC Candidate Status"] = np.where(
    df_qc_audit["genome_quality"].eq("Good"),
    "Keep",
    "Exclude"
)

candidate_counts = (
    df_qc_audit["QC Candidate Status"]
    .value_counts()
)

print("\n" + "-" * 70)
print("CANDIDATE QC FILTER SUMMARY")
print("-" * 70)

display(
    candidate_counts
    .rename_axis("QC Candidate Status")
    .reset_index(name="Genome Count")
)

print(
    f"\nCandidate genomes retained : "
    f"{(df_qc_audit['QC Candidate Status'] == 'Keep').sum():,}"
)

print(
    f"Candidate genomes excluded : "
    f"{(df_qc_audit['QC Candidate Status'] == 'Exclude').sum():,}"
)

print(
    f"Retention rate             : "
    f"{(df_qc_audit['QC Candidate Status'] == 'Keep').mean() * 100:.2f}%"
)


# ------------------------------------------------------------
# 3. Phenotype distribution trước và sau candidate QC filter
# ------------------------------------------------------------

before_qc = (
    df_qc_audit["Resistant Phenotype"]
    .value_counts()
    .rename("Before QC")
)

after_qc = (
    df_qc_audit.loc[
        df_qc_audit["QC Candidate Status"] == "Keep",
        "Resistant Phenotype"
    ]
    .value_counts()
    .rename("After QC")
)

excluded_qc = (
    df_qc_audit.loc[
        df_qc_audit["QC Candidate Status"] == "Exclude",
        "Resistant Phenotype"
    ]
    .value_counts()
    .rename("Excluded")
)

phenotype_qc_impact = (
    pd.concat(
        [
            before_qc,
            after_qc,
            excluded_qc
        ],
        axis=1
    )
    .fillna(0)
    .astype(int)
)

phenotype_qc_impact["Retention (%)"] = (
    phenotype_qc_impact["After QC"]
    / phenotype_qc_impact["Before QC"]
    * 100
).round(2)

print("\n" + "-" * 70)
print("PHENOTYPE IMPACT OF CANDIDATE QC FILTER")
print("-" * 70)

display(phenotype_qc_impact)


# ------------------------------------------------------------
# 4. Kiểm tra tỷ lệ Poor theo phenotype
# ------------------------------------------------------------

poor_by_phenotype = (
    df_qc_audit
    .groupby("Resistant Phenotype")["QC Candidate Status"]
    .apply(
        lambda x:
            (x == "Exclude").mean() * 100
    )
    .round(2)
    .rename("Poor Genome (%)")
)

print("\n" + "-" * 70)
print("POOR GENOME RATE BY PHENOTYPE")
print("-" * 70)

display(
    poor_by_phenotype
    .reset_index()
)


# ------------------------------------------------------------
# 5. Chuẩn hóa genome quality flags
# ------------------------------------------------------------

def normalize_quality_flags(value):

    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    return [str(value)]


df_qc_audit["Quality Flag List"] = (
    df_qc_audit["genome_quality_flags"]
    .apply(normalize_quality_flags)
)


# ------------------------------------------------------------
# 6. Đếm từng loại quality flag riêng lẻ
# ------------------------------------------------------------

flag_counts = (
    df_qc_audit.loc[
        df_qc_audit["QC Candidate Status"] == "Exclude",
        "Quality Flag List"
    ]
    .explode()
    .dropna()
    .value_counts()
    .rename_axis("Quality Flag")
    .reset_index(name="Genome Count")
)

print("\n" + "-" * 70)
print("QUALITY FLAG FREQUENCY AMONG POOR GENOMES")
print("-" * 70)

display(flag_counts)


# ------------------------------------------------------------
# 7. Hiển thị toàn bộ candidate excluded genomes
# ------------------------------------------------------------

excluded_columns = [
    "genome_id",
    "genome_name",
    "Resistant Phenotype",
    "genome_quality",
    "genome_quality_flags",
    "checkm_completeness",
    "checkm_contamination",
    "genome_length",
    "contigs",
    "contig_n50",
    "gc_content",
    "cds",
    "coarse_consistency",
    "fine_consistency"
]

df_qc_excluded_candidates = (
    df_qc_audit.loc[
        df_qc_audit["QC Candidate Status"] == "Exclude",
        excluded_columns
    ]
    .sort_values(
        [
            "checkm_contamination",
            "contigs"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("\n" + "-" * 70)
print("CANDIDATE EXCLUDED GENOMES")
print("-" * 70)

display(df_qc_excluded_candidates)


# ------------------------------------------------------------
# 8. Final audit status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("QC FILTER IMPACT AUDIT COMPLETE")
print("=" * 70)

print(
    "NOTE: No genomes have been permanently removed yet."
)

BV-BRC GENOME QUALITY FILTER IMPACT AUDIT
Genome quality distribution:


,Genome Quality,Genome Count
0,Good,4233
1,Poor,37



----------------------------------------------------------------------
CANDIDATE QC FILTER SUMMARY
----------------------------------------------------------------------


,QC Candidate Status,Genome Count
0,Keep,4233
1,Exclude,37



Candidate genomes retained : 4,233
Candidate genomes excluded : 37
Retention rate             : 99.13%

----------------------------------------------------------------------
PHENOTYPE IMPACT OF CANDIDATE QC FILTER
----------------------------------------------------------------------


...


----------------------------------------------------------------------
QUALITY FLAG FREQUENCY AMONG POOR GENOMES
----------------------------------------------------------------------


,Quality Flag,Genome Count
0,Too many contigs,24
1,High CheckM contamination score,11
2,Low contig N50,4
3,Low CheckM completeness score,1
4,Genome too short,1
5,Abnormal CDS ratio,1



----------------------------------------------------------------------
CANDIDATE EXCLUDED GENOMES
----------------------------------------------------------------------


,genome_id,genome_name,Resistant Phenotype,genome_quality,genome_quality_flags,checkm_completeness,checkm_contamination,genome_length,contigs,contig_n50,gc_content,cds,coarse_consistency,fine_consistency
0,573.31080,Klebsiella pneumoniae strain WU21,Resistant,Poor,[Too many contigs],99.44,51.3,11147041,1715,14403,56.560265,11674,94.39,39.90
1,573.12957,Klebsiella pneumoniae 3252,Susceptible,Poor,[Too many contigs],100.00,50.7,7355114,5155,99606,55.740000,10740,98.00,55.10
2,573.12987,Klebsiella pneumoniae 3386,Susceptible,Poor,[Too many contigs],100.00,48.7,8354191,3848,100473,51.810000,10217,97.70,51.70
3,573.15570,Klebsiella pneumoniae strain KSB1_4J,Susceptible,Poor,[Too many contigs],98.30,48.3,9866087,1611,34682,54.280000,11229,98.50,52.70
4,573.13835,Klebsiella pneumoniae KPN2005ec,Susceptible,Poor,[Too many contigs],90.80,47.8,8750302,2412,5163,57.550000,10212,92.20,52.30
5,573.15450,Klebsiella pneumoniae strain 10315_6#32,Susceptible,Poor,[High CheckM contamination score],99.80,47.5,10834068,593,46275,53.600000,11113,97.60,51.40
6,573.13822,Klebsiella pneumoniae KPN1989ec,Susceptible,Poor,[Too many contigs],100.00,40.7,8614068,1747,15828,54.740000,10163,98.70,66.30
7,573.13345,Klebsiella pneumoniae KPN139ec,Susceptible,Poor,[Too many contigs],100.00,40.3,8862200,1209,31009,50.760000,9139,98.40,54.50
8,573.13353,Klebsiella pneumoniae KPN140ec,Susceptible,Poor,[Too many contigs],100.00,39.8,8883036,1128,31829,50.700000,9106,98.70,54.70
9,573.14330,Klebsiella pneumoniae KPN702ec,Resistant,Poor,[Too many contigs],77.70,37.7,6615681,1744,32046,46.350000,7953,90.60,60.50



QC FILTER IMPACT AUDIT COMPLETE
NOTE: No genomes have been permanently removed yet.


# Cell 9 — QC Filter Bias Statistical Audit

In [ ]:
# ============================================================
# CELL 9 — QC FILTER BIAS STATISTICAL AUDIT
# ============================================================

from scipy.stats import fisher_exact

print("=" * 70)
print("QC FILTER BIAS STATISTICAL AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Tạo contingency table:
#    phenotype × QC candidate status
# ------------------------------------------------------------

qc_contingency = pd.crosstab(
    df_qc_audit["Resistant Phenotype"],
    df_qc_audit["QC Candidate Status"]
)

# Đảm bảo thứ tự cột rõ ràng
for col in ["Keep", "Exclude"]:
    if col not in qc_contingency.columns:
        qc_contingency[col] = 0

qc_contingency = qc_contingency[
    ["Keep", "Exclude"]
]

print("Phenotype × QC candidate status:")

display(qc_contingency)


# ------------------------------------------------------------
# 2. Tính tỷ lệ exclusion theo phenotype
# ------------------------------------------------------------

qc_exclusion_rates = (
    qc_contingency["Exclude"]
    / qc_contingency.sum(axis=1)
    * 100
).round(4)

qc_exclusion_summary = pd.DataFrame({
    "Total Genomes":
        qc_contingency.sum(axis=1),

    "Excluded Genomes":
        qc_contingency["Exclude"],

    "Exclusion Rate (%)":
        qc_exclusion_rates
})

print("\n" + "-" * 70)
print("QC EXCLUSION RATE BY PHENOTYPE")
print("-" * 70)

display(qc_exclusion_summary)


# ------------------------------------------------------------
# 3. Fisher's exact test
#
# Table order:
# Resistant   -> Exclude, Keep
# Susceptible -> Exclude, Keep
# ------------------------------------------------------------

fisher_table = [
    [
        qc_contingency.loc["Resistant", "Exclude"],
        qc_contingency.loc["Resistant", "Keep"]
    ],
    [
        qc_contingency.loc["Susceptible", "Exclude"],
        qc_contingency.loc["Susceptible", "Keep"]
    ]
]

odds_ratio, fisher_p_value = fisher_exact(
    fisher_table,
    alternative="two-sided"
)


# ------------------------------------------------------------
# 4. Absolute difference in exclusion rate
# ------------------------------------------------------------

resistant_exclusion_rate = (
    qc_exclusion_rates.loc["Resistant"]
)

susceptible_exclusion_rate = (
    qc_exclusion_rates.loc["Susceptible"]
)

absolute_rate_difference = abs(
    resistant_exclusion_rate
    - susceptible_exclusion_rate
)


# ------------------------------------------------------------
# 5. Class distribution before vs candidate QC filter
# ------------------------------------------------------------

before_distribution = (
    df_qc_audit["Resistant Phenotype"]
    .value_counts(normalize=True)
    .mul(100)
)

after_distribution = (
    df_qc_audit.loc[
        df_qc_audit["QC Candidate Status"] == "Keep",
        "Resistant Phenotype"
    ]
    .value_counts(normalize=True)
    .mul(100)
)

class_distribution_comparison = pd.concat(
    [
        before_distribution.rename("Before QC (%)"),
        after_distribution.rename("After QC (%)")
    ],
    axis=1
).round(4)

class_distribution_comparison[
    "Change (percentage points)"
] = (
    class_distribution_comparison["After QC (%)"]
    - class_distribution_comparison["Before QC (%)"]
).round(4)

print("\n" + "-" * 70)
print("CLASS DISTRIBUTION BEFORE VS AFTER CANDIDATE QC")
print("-" * 70)

display(class_distribution_comparison)


# ------------------------------------------------------------
# 6. Statistical summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("QC FILTER BIAS TEST SUMMARY")
print("=" * 70)

print(
    f"Resistant exclusion rate   : "
    f"{resistant_exclusion_rate:.4f}%"
)

print(
    f"Susceptible exclusion rate : "
    f"{susceptible_exclusion_rate:.4f}%"
)

print(
    f"Absolute rate difference   : "
    f"{absolute_rate_difference:.4f} percentage points"
)

print(
    f"Fisher odds ratio          : "
    f"{odds_ratio:.4f}"
)

print(
    f"Fisher exact p-value       : "
    f"{fisher_p_value:.6f}"
)

print("\nInterpretation rule:")

print(
    "- p < 0.05 indicates that QC exclusion rates differ "
    "statistically between phenotypes."
)

print(
    "- Statistical significance should be interpreted together "
    "with the absolute exclusion rate and class-distribution change."
)

print(
    "\nNOTE: No genomes have been permanently removed yet."
)

QC FILTER BIAS STATISTICAL AUDIT
Phenotype × QC candidate status:


QC Candidate Status,Keep,Exclude
Resistant Phenotype,,
Resistant,1603,6
Susceptible,2630,31



----------------------------------------------------------------------
QC EXCLUSION RATE BY PHENOTYPE
----------------------------------------------------------------------


,Total Genomes,Excluded Genomes,Exclusion Rate (%)
Resistant Phenotype,,,
Resistant,1609,6,0.3729
Susceptible,2661,31,1.1650



----------------------------------------------------------------------
CLASS DISTRIBUTION BEFORE VS AFTER CANDIDATE QC
----------------------------------------------------------------------


,Before QC (%),After QC (%),Change (percentage points)
Resistant Phenotype,,,
Susceptible,62.3185,62.1309,-0.1876
Resistant,37.6815,37.8691,0.1876



QC FILTER BIAS TEST SUMMARY
Resistant exclusion rate   : 0.3729%
Susceptible exclusion rate : 1.1650%
Absolute rate difference   : 0.7921 percentage points
Fisher odds ratio          : 0.3175
Fisher exact p-value       : 0.005988

Interpretation rule:
- p < 0.05 indicates that QC exclusion rates differ statistically between phenotypes.
- Statistical significance should be interpreted together with the absolute exclusion rate and class-distribution change.

NOTE: No genomes have been permanently removed yet.


# Cell 10 — Apply Genome Quality Filter & Create QC-Qualified Cohort

In [ ]:
# ============================================================
# CELL 10 — APPLY GENOME QUALITY FILTER
#            & CREATE QC-QUALIFIED COHORT
# ============================================================

import os

print("=" * 70)
print("APPLY GENOME QUALITY FILTER & CREATE QC-QUALIFIED COHORT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Chính thức xác định genome được giữ / loại
#
# Quy tắc:
# BV-BRC genome_quality == "Good"  -> Keep
# BV-BRC genome_quality == "Poor"  -> Exclude
#
# Không sử dụng phenotype để quyết định QC status.
# ------------------------------------------------------------

df_qc_clean = (
    df_qc_audit.loc[
        df_qc_audit["genome_quality"].eq("Good")
    ]
    .copy()
    .reset_index(drop=True)
)

df_qc_excluded = (
    df_qc_audit.loc[
        df_qc_audit["genome_quality"].ne("Good")
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. Tạo QC-qualified manifest
# ------------------------------------------------------------

qualified_genome_ids = set(
    df_qc_clean["genome_id"].astype(str)
)

df_manifest_qc = (
    df_manifest.loc[
        df_manifest["Genome ID String"]
        .astype(str)
        .isin(qualified_genome_ids)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Kiểm tra số lượng
# ------------------------------------------------------------

original_count = len(df_manifest)
retained_count = len(df_manifest_qc)
excluded_count = len(df_qc_excluded)

retention_rate = (
    retained_count
    / original_count
    * 100
)

print(f"Original genomes       : {original_count:,}")
print(f"Retained genomes       : {retained_count:,}")
print(f"Excluded genomes       : {excluded_count:,}")
print(f"Retention rate         : {retention_rate:.2f}%")


# ------------------------------------------------------------
# 4. Kiểm tra target distribution sau QC
# ------------------------------------------------------------

target_after_qc = (
    df_manifest_qc["Resistant Phenotype"]
    .value_counts()
)

target_after_qc_pct = (
    df_manifest_qc["Resistant Phenotype"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_after_qc_summary = pd.DataFrame({
    "Genome Count": target_after_qc,
    "Percentage (%)": target_after_qc_pct
})

print("\n" + "-" * 70)
print("TARGET DISTRIBUTION AFTER QC")
print("-" * 70)

display(target_after_qc_summary)


# ------------------------------------------------------------
# 5. Kiểm tra integrity của QC-qualified manifest
# ------------------------------------------------------------

missing_genome_ids_qc = (
    df_manifest_qc["Genome ID String"]
    .isna()
    .sum()
)

duplicate_genome_ids_qc = (
    df_manifest_qc["Genome ID String"]
    .duplicated()
    .sum()
)

unexpected_targets_qc = (
    set(
        df_manifest_qc["Resistant Phenotype"]
        .dropna()
        .unique()
    )
    - {
        "Resistant",
        "Susceptible"
    }
)

poor_genomes_remaining = (
    df_qc_clean["genome_quality"]
    .ne("Good")
    .sum()
)

print("\n" + "-" * 70)
print("QC-QUALIFIED COHORT INTEGRITY")
print("-" * 70)

print(
    f"Missing Genome IDs      : "
    f"{missing_genome_ids_qc:,}"
)

print(
    f"Duplicate Genome IDs    : "
    f"{duplicate_genome_ids_qc:,}"
)

print(
    f"Unexpected targets      : "
    f"{sorted(unexpected_targets_qc) if unexpected_targets_qc else 'None'}"
)

print(
    f"Poor genomes remaining  : "
    f"{poor_genomes_remaining:,}"
)


# ------------------------------------------------------------
# 6. Final assertions
# ------------------------------------------------------------

assert retained_count == 4233, \
    "Unexpected number of retained genomes."

assert excluded_count == 37, \
    "Unexpected number of excluded genomes."

assert missing_genome_ids_qc == 0, \
    "Missing Genome ID detected after QC."

assert duplicate_genome_ids_qc == 0, \
    "Duplicate Genome ID detected after QC."

assert len(unexpected_targets_qc) == 0, \
    "Unexpected phenotype label detected."

assert poor_genomes_remaining == 0, \
    "Poor-quality genome remains in QC-qualified cohort."

assert (
    set(df_manifest_qc["Genome ID String"].astype(str))
    == set(df_qc_clean["genome_id"].astype(str))
), "Genome IDs do not match between manifest and QC table."


# ------------------------------------------------------------
# 7. Export QC outputs
# ------------------------------------------------------------

OUTPUT_DIR = "/content/processed_data"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

qc_manifest_path = os.path.join(
    OUTPUT_DIR,
    "genome_manifest_qc.csv"
)

qc_metadata_path = os.path.join(
    OUTPUT_DIR,
    "genome_qc_metadata.csv"
)

qc_excluded_path = os.path.join(
    OUTPUT_DIR,
    "genome_qc_excluded.csv"
)

df_manifest_qc.to_csv(
    qc_manifest_path,
    index=False
)

df_qc_clean.to_csv(
    qc_metadata_path,
    index=False
)

df_qc_excluded.to_csv(
    qc_excluded_path,
    index=False
)


# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL QC FILTER SUMMARY")
print("=" * 70)

print("✓ BV-BRC Poor-quality genomes excluded.")
print(f"✓ QC-qualified genomes : {len(df_manifest_qc):,}")

print("\nExported files:")
print(f"- {qc_manifest_path}")
print(f"- {qc_metadata_path}")
print(f"- {qc_excluded_path}")

APPLY GENOME QUALITY FILTER & CREATE QC-QUALIFIED COHORT
Original genomes       : 4,270
Retained genomes       : 4,233
Excluded genomes       : 37
Retention rate         : 99.13%

----------------------------------------------------------------------
TARGET DISTRIBUTION AFTER QC
----------------------------------------------------------------------


,Genome Count,Percentage (%)
Resistant Phenotype,,
Susceptible,2630,62.13
Resistant,1603,37.87



----------------------------------------------------------------------
QC-QUALIFIED COHORT INTEGRITY
----------------------------------------------------------------------
Missing Genome IDs      : 0
Duplicate Genome IDs    : 0
Unexpected targets      : None
Poor genomes remaining  : 0

FINAL QC FILTER SUMMARY
✓ BV-BRC Poor-quality genomes excluded.
✓ QC-qualified genomes : 4,233

Exported files:
- /content/processed_data/genome_manifest_qc.csv
- /content/processed_data/genome_qc_metadata.csv
- /content/processed_data/genome_qc_excluded.csv
